# 01 — Exploratory Data Analysis

**Goal:** Understand the Telco Customer Churn dataset well enough to inform feature choices, baseline expectations, and the Week 2 cost-sensitive analysis.

**Key questions answered here:**
1. What is the overall churn rate? (Sets our class-imbalance expectation.)
2. Which contract, service, and billing features correlate most strongly with churn?
3. What is the monthly recurring revenue (MRR) at risk from churners?

In [ ]:
import sys
from pathlib import Path

# Make the project's src/ package importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data_loader import load_clean

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

## 1. Load the data

`load_clean` downloads via kagglehub (cached after first run), coerces `TotalCharges` to numeric, drops the ~11 tenure=0 rows with blank charges, and normalizes `SeniorCitizen` to Yes/No.

In [ ]:
df = load_clean()
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

## 2. Target distribution

Churn is imbalanced (~27% positive class). This matters for model choice: accuracy is a misleading metric here, so we will lead with **ROC-AUC** and **recall on the churn class**.

In [ ]:
churn_rate = (df["Churn"] == "Yes").mean()
print(f"Overall churn rate: {churn_rate:.1%}")

fig, ax = plt.subplots(figsize=(4, 3))
sns.countplot(data=df, x="Churn", order=["No", "Yes"], ax=ax)
ax.set_title("Target distribution")
plt.show()

## 3. Numeric features vs. churn

Three numeric columns: `tenure`, `MonthlyCharges`, `TotalCharges`. Hypothesis: short tenure + high monthly charges → higher churn risk.

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.kdeplot(data=df, x=col, hue="Churn", common_norm=False, ax=ax)
    ax.set_title(f"{col} by churn")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("Churn")[numeric_cols].agg(["mean", "median"]).round(2)

## 4. Categorical features: churn rate by group

We plot churn rate (not raw counts) per level so imbalanced groups don't dominate.

In [ ]:
cat_cols = [
    "Contract", "InternetService", "PaymentMethod", "PaperlessBilling",
    "OnlineSecurity", "TechSupport", "SeniorCitizen", "Partner", "Dependents",
]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, col in zip(axes.flat, cat_cols):
    rates = df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values()
    rates.plot.barh(ax=ax, color="steelblue")
    ax.axvline(churn_rate, color="red", linestyle="--", alpha=0.6, label="overall")
    ax.set_title(f"Churn rate by {col}")
    ax.set_xlabel("churn rate")
    ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

**Observations to revisit in modeling:**
- Month-to-month contracts churn at a drastically higher rate than 1- or 2-year contracts.
- Fiber optic customers churn more than DSL — worth investigating whether price, quality, or both is the driver.
- Electronic check payment is associated with elevated churn.
- Lack of `OnlineSecurity` and `TechSupport` add-ons correlates with churn.

## 5. Revenue at risk

The business question is not just *who* churns but *how much revenue* churns. Sum of `MonthlyCharges` for current churners = MRR we'd lose if we took no action. This becomes the denominator for Week 2's ROI analysis.

In [ ]:
total_mrr = df["MonthlyCharges"].sum()
mrr_at_risk = df.loc[df["Churn"] == "Yes", "MonthlyCharges"].sum()
print(f"Total MRR in snapshot:  ${total_mrr:>12,.2f}")
print(f"MRR from churners:      ${mrr_at_risk:>12,.2f}  ({mrr_at_risk / total_mrr:.1%})")

In [ ]:
# Revenue concentration: are churners disproportionately high- or low-spend?
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", order=["No", "Yes"], ax=ax)
ax.set_title("Monthly charges: stayers vs. churners")
plt.show()

## 6. Takeaways into modeling

1. **Class imbalance (~27% churn)** — use `class_weight="balanced"` on logistic regression and `scale_pos_weight` on XGBoost. Evaluate with AUC and recall, not accuracy.
2. **Contract type is the dominant signal** — expect it to top the feature importance chart.
3. **Churners are higher-MRR on average** — retention is financially worthwhile even before optimizing threshold.
4. **`TotalCharges` is nearly collinear with `tenure × MonthlyCharges`** — may be redundant for a linear model, but XGBoost should handle it fine.